In [1]:
# The actual Dual-Axis data now we are using
# Using the Protests and Riots columns as events and Violence against civilians
# column as fatalities.

import pandas as pd

print("=== Chart 7: Dual-Axis Line Chart (Protests vs Fatalities in Iran) ===")

# 1. Load the Middle East data
file_name = 'Middle-East_aggregated_data_up_to-2026-01-24.xlsx'
df_me = pd.read_excel(file_name)

# 2. Create year and month columns for accurate aggregation
# We convert the date to "Year-Month" format (e.g., 2022-09)
df_me['DATE'] = pd.to_datetime(df_me['WEEK'])
df_me['YEAR_MONTH'] = df_me['DATE'].dt.to_period('M')

# 3. Filter Iran's data in the desired time frame (2017 to 2025)
df_iran = df_me[(df_me['COUNTRY'] == 'Iran') &
                (df_me['DATE'].dt.year >= 2017) &
                (df_me['DATE'].dt.year <= 2025)].copy()

# 4. Extract protest data (Protests & Riots)
# For protests, we count the number of "EVENTS"
df_protests = df_iran[df_iran['EVENT_TYPE'].isin(['Protests', 'Riots'])]
protest_counts = df_protests.groupby('YEAR_MONTH')['EVENTS'].sum().reset_index()
protest_counts.rename(columns={'EVENTS': 'Protest_Events'}, inplace=True)

# 5. Extract repression data (Violence against civilians)
# For repression, we count the number of "FATALITIES"
df_violence = df_iran[df_iran['EVENT_TYPE'] == 'Violence against civilians']
violence_fatalities = df_violence.groupby('YEAR_MONTH')['FATALITIES'].sum().reset_index()
violence_fatalities.rename(columns={'FATALITIES': 'Civilian_Fatalities'}, inplace=True)

# 6. Merge the two dataframes based on the month
# We use a left merge so that if there was a protest in a month but no fatalities, fatalities are set to zero
df_dual_axis = pd.merge(protest_counts, violence_fatalities, on='YEAR_MONTH', how='left').fillna(0)

# 7. Convert YEAR_MONTH to a string
df_dual_axis['YEAR_MONTH'] = df_dual_axis['YEAR_MONTH'].astype(str)

# 8. Save the file
output_file_7 = 'chart7_dual_axis.csv'
df_dual_axis.to_csv(output_file_7, index=False)

print("\n Chart 7 Data Ready! Examples of the bloodiest months:")
print(df_dual_axis.sort_values(by='Civilian_Fatalities', ascending=False).head(5))
print(f"\nData saved to: {output_file_7}")

=== Chart 7: Dual-Axis Line Chart (Protests vs Fatalities in Iran) ===

 Chart 7 Data Ready! Examples of the bloodiest months:
   YEAR_MONTH  Protest_Events  Civilian_Fatalities
69    2022-10             770                 48.0
43    2020-08             271                 27.0
70    2022-11             426                 24.0
74    2023-03             209                 23.0
71    2022-12             304                 20.0

Data saved to: chart7_dual_axis.csv


In [3]:
# Using the Protests and Riots columns both as events and fatalities.

import pandas as pd

print("=== Chart 7: Dual-Axis Line Chart (Protest Volume vs Fatalities in Iran) ===")

# 1. Load the Middle East data
file_name = 'Middle-East_aggregated_data_up_to-2026-01-24.xlsx'
df_me = pd.read_excel(file_name)

# 2. Create Year-Month column for aggregation
df_me['WEEK'] = pd.to_datetime(df_me['WEEK'])
df_me['YEAR_MONTH'] = df_me['WEEK'].dt.to_period('M')

# 3. Filter for Iran (2017 to 2025)
df_iran = df_me[(df_me['COUNTRY'] == 'Iran') &
                (df_me['WEEK'].dt.year >= 2017) &
                (df_me['WEEK'].dt.year <= 2025)].copy()

# 4. Filter for only Protest and Riot events
df_protests_riots = df_iran[df_iran['EVENT_TYPE'].isin(['Protests', 'Riots'])]

# 5. Extract Axis 1: Number of Protest/Riot EVENTS (Volume)
protest_counts = df_protests_riots.groupby('YEAR_MONTH')['EVENTS'].sum().reset_index()
protest_counts.rename(columns={'EVENTS': 'Protest_Events'}, inplace=True)

# 6. Extract Axis 2: Sum of FATALITIES from those Protests/Riots (Violence/Crackdown)
protest_fatalities = df_protests_riots.groupby('YEAR_MONTH')['FATALITIES'].sum().reset_index()
protest_fatalities.rename(columns={'FATALITIES': 'Protest_Fatalities'}, inplace=True)

# 7. Merge the two dataframes
df_dual_axis = pd.merge(protest_counts, protest_fatalities, on='YEAR_MONTH', how='outer').fillna(0)

# 8. Convert YEAR_MONTH to string for JavaScript/Charting tools
df_dual_axis['YEAR_MONTH'] = df_dual_axis['YEAR_MONTH'].astype(str)

# Sort chronologically just in case
df_dual_axis = df_dual_axis.sort_values('YEAR_MONTH')

# 9. Save to CSV
output_file_7 = 'chart7_dual_axis.csv'
df_dual_axis.to_csv(output_file_7, index=False)

print("\n Data for Chart 7 is ready! Here are the top 5 deadliest months:")
print(df_dual_axis.sort_values(by='Protest_Fatalities', ascending=False).head(5))

=== Chart 7: Dual-Axis Line Chart (Protest Volume vs Fatalities in Iran) ===

 Data for Chart 7 is ready! Here are the top 5 deadliest months:
   YEAR_MONTH  Protest_Events  Protest_Fatalities
34    2019-11             334                 387
68    2022-09             383                 231
70    2022-11             426                 134
69    2022-10             770                 104
38    2020-03              63                  33


In [4]:
import pandas as pd

print("=== Chart 8: Ridgeline Plot Preprocessing (Weekly Fatalities) ===")

# 1. Load the Middle East data
file_name = 'Middle-East_aggregated_data_up_to-2026-01-24.xlsx'
df_me = pd.read_excel(file_name)

# 2. Extract the date
df_me['DATE'] = pd.to_datetime(df_me['WEEK'])
df_iran = df_me[(df_me['COUNTRY'] == 'Iran') &
                (df_me['DATE'].dt.year >= 2017) &
                (df_me['DATE'].dt.year <= 2025)].copy()

# 3. Create year and "week number" columns (to have 52 points per year for drawing smooth curves)
df_iran['YEAR'] = df_iran['DATE'].dt.year
df_iran['WEEK_NUM'] = df_iran['DATE'].dt.isocalendar().week

# 4. Filter protest and violent events
df_target = df_iran[df_iran['EVENT_TYPE'].isin(['Violence against civilians', 'Protests', 'Riots'])]

# 5. Aggregate data based on "year" and "week"
df_ridge = df_target.groupby(['YEAR', 'WEEK_NUM']).agg(
    Fatalities=('FATALITIES', 'sum'),
    Events=('EVENTS', 'sum')
).reset_index()

# 6. Fill empty weeks with zero
# We create a complete grid of all years and 52 weeks
all_years = range(2017, 2026)
all_weeks = range(1, 54)   # Up to week 53 for leap years
mux = pd.MultiIndex.from_product([all_years, all_weeks], names=['YEAR', 'WEEK_NUM'])
df_ridge = df_ridge.set_index(['YEAR', 'WEEK_NUM']).reindex(mux, fill_value=0).reset_index()

# 7. Save the file
output_file_8 = 'chart8_ridgeline.csv'
df_ridge.to_csv(output_file_8, index=False)

print("\n Chart 8 Data Ready!")
print(df_ridge[df_ridge['YEAR'] == 2022].sort_values(by='Fatalities', ascending=False).head(5))

=== Chart 8: Ridgeline Plot Preprocessing (Weekly Fatalities) ===

 Chart 8 Data Ready!
     YEAR  WEEK_NUM  Fatalities  Events
301  2022        37         124     223
302  2022        38         113     114
309  2022        45          89     166
306  2022        42          62     220
310  2022        46          47     127
